In [4]:
import ssl
# --- Fix for Windows SSL/cert-store errors when MediaPipe tries to download a
# model file it doesn't have cached yet (SSLError: [ASN1: NOT_ENOUGH_DATA]).
# Tries certifi's certificate bundle first; falls back to an unverified
# context (only used for this local model download) if certifi is missing.
try:
    import certifi
    ssl._create_default_https_context = lambda: ssl.create_default_context(cafile=certifi.where())
except ImportError:
    ssl._create_default_https_context = ssl._create_unverified_context

import cv2
import mediapipe as mp
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from torch.optim import Adam
from torch.optim.lr_scheduler import CosineAnnealingLR
from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import confusion_matrix, accuracy_score, classification_report, f1_score
import os
import glob
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# =============================================================================
# CONFIGURATION
# =============================================================================
VIDEO_DIR = r"C:\Users\admin\Downloads\A Multi-View Raw Video Dataset of Seven Fitness Ex\A Multi-View Raw Video Dataset of Seven Fitness Ex\Dataset Exercise Quality-wise\squat\New folder"

# ===== FIXED LABELLING STRATEGY =====
LABEL_STRATEGY = "threshold"
SI_THRESHOLD = 10.0

# Other parameters
SEQ_LEN = 45
STRIDE = 10
BATCH_SIZE = 32
EPOCHS = 20
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 1e-3
SEED = 42
PATIENCE = 8
EARLY_STOP_MIN_EPOCH = 10

# Which model(s) to train
MODELS_TO_TRAIN = ["CTR-GCN", "ST-GCN"]

# Feature cache
CACHE_FILE = os.path.join(os.path.expanduser("~"), "Downloads", "squat_features_cache_1 (1).pt")
FORCE_REEXTRACT = False

# Cross-validation settings
RUN_CV = True          # Set to True to run cross-validation, False for single train
N_FOLDS = 5            # Number of folds for cross-validation

# Graph adjacency for 17 joints
ADJ = [
    (0,1),(0,2),(1,3),(2,4), (5,6),(5,7),(7,9),(6,8),(8,10),
    (11,12),(11,13),(13,15),(12,14),(14,16), (5,11),(6,12)
]

def set_seed(seed=SEED):
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

if device.type == 'cpu':
    n_threads = os.cpu_count() or 4
    torch.set_num_threads(n_threads)
    print(f"Using device: {device} | torch threads set to {n_threads}")
else:
    print(f"Using device: {device}")

def get_adj_matrix():
    A = np.eye(17)
    for i, j in ADJ:
        A[i, j] = A[j, i] = 1
    D = np.diag(np.sum(A, axis=1)**-0.5)
    A_norm = D @ A @ D
    return torch.FloatTensor(A_norm)

# =============================================================================
# SKELETON UTILS & PROCESSOR
# =============================================================================
class SkeletonUtils:
    @staticmethod
    def normalize(coords):
        mid_hip = (coords[:, 11, :] + coords[:, 12, :]) / 2
        coords = coords - mid_hip[:, np.newaxis, :]
        shoulder_mid = (coords[:, 0, :] + coords[:, 1, :]) / 2
        hip_mid = (coords[:, 11, :] + coords[:, 12, :]) / 2
        body_scale = np.linalg.norm(shoulder_mid - hip_mid, axis=-1).mean()
        if body_scale > 1e-6:
            coords = coords / body_scale
        mean = coords.mean(axis=0, keepdims=True)
        std = coords.std(axis=0, keepdims=True) + 1e-8
        coords = (coords - mean) / std
        return coords

    @staticmethod
    def calculate_angle(a, b, c):
        a, b, c = np.array(a), np.array(b), np.array(c)
        ba = a - b
        bc = c - b
        cos_angle = np.dot(ba, bc) / (np.linalg.norm(ba) * np.linalg.norm(bc) + 1e-8)
        angle = np.arccos(np.clip(cos_angle, -1.0, 1.0))
        return np.degrees(angle)

    @staticmethod
    def augment_batch(x):
        x = x.clone()
        B = x.size(0)
        pos = x[:, 0:3]
        vel = x[:, 3:6]
        acc = x[:, 6:9]
        jerk = x[:, 9:12]

        x += torch.randn_like(x) * 0.02

        do_rot = torch.rand(B) > 0.5
        if do_rot.any():
            angles = (torch.rand(B) - 0.5) * 0.3
            c, s = torch.cos(angles), torch.sin(angles)
            zeros, ones = torch.zeros(B), torch.ones(B)
            rot = torch.stack([
                torch.stack([c, zeros, s], dim=1),
                torch.stack([zeros, ones, zeros], dim=1),
                torch.stack([-s, zeros, c], dim=1),
            ], dim=1).to(x.device)

            idx = do_rot.nonzero(as_tuple=True)[0]
            pos_sub = pos[idx]
            vel_sub = vel[idx]
            acc_sub = acc[idx]
            jerk_sub = jerk[idx]
            b, _, T, V = pos_sub.shape
            pos_flat = pos_sub.view(b, 3, -1)
            vel_flat = vel_sub.view(b, 3, -1)
            acc_flat = acc_sub.view(b, 3, -1)
            jerk_flat = jerk_sub.view(b, 3, -1)

            rot_sub = rot[idx]
            pos_rot = torch.matmul(rot_sub, pos_flat)
            vel_rot = torch.matmul(rot_sub, vel_flat)
            acc_rot = torch.matmul(rot_sub, acc_flat)
            jerk_rot = torch.matmul(rot_sub, jerk_flat)

            pos[idx] = pos_rot.view(b, 3, T, V)
            vel[idx] = vel_rot.view(b, 3, T, V)
            acc[idx] = acc_rot.view(b, 3, T, V)
            jerk[idx] = jerk_rot.view(b, 3, T, V)

        x = torch.cat([pos, vel, acc, jerk], dim=1)
        scale = 1 + (torch.rand(B, 1, 1, 1) - 0.5) * 0.2
        x = x * scale
        return x

class SquatProcessor:
    def __init__(self):
        self.pose = mp.solutions.pose.Pose(
            static_image_mode=False,
            model_complexity=1,
            min_detection_confidence=0.5
        )

    def get_si(self, left_angle, right_angle):
        return ((left_angle - right_angle) / (0.5 * (left_angle + right_angle) + 1e-6)) * 100

    def process_video(self, path):
        cap = cv2.VideoCapture(path)
        coords = []
        frame_data = []

        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break
            img = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            res = self.pose.process(img)

            if res.pose_world_landmarks:
                wlm = res.pose_world_landmarks.landmark
                pts = [[lm.x, lm.y, lm.z] for lm in wlm[12:29]]
                coords.append(pts)

                left_knee = SkeletonUtils.calculate_angle(
                    [wlm[23].x, wlm[23].y, wlm[23].z],
                    [wlm[25].x, wlm[25].y, wlm[25].z],
                    [wlm[27].x, wlm[27].y, wlm[27].z]
                )
                right_knee = SkeletonUtils.calculate_angle(
                    [wlm[24].x, wlm[24].y, wlm[24].z],
                    [wlm[26].x, wlm[26].y, wlm[26].z],
                    [wlm[28].x, wlm[28].y, wlm[28].z]
                )
                si = self.get_si(left_knee, right_knee)

                frame_data.append({
                    'si': si,
                    'left_knee': left_knee,
                    'right_knee': right_knee,
                    'avg_knee': (left_knee + right_knee) / 2,
                    'label': 1 if si > SI_THRESHOLD else 0
                })

        cap.release()
        if len(coords) < SEQ_LEN:
            return None, None, None

        coords = SkeletonUtils.normalize(np.array(coords))

        avg_si = np.mean([d['si'] for d in frame_data])
        avg_left_knee = np.mean([d['left_knee'] for d in frame_data])
        avg_right_knee = np.mean([d['right_knee'] for d in frame_data])
        avg_knee = (avg_left_knee + avg_right_knee) / 2

        frame_labels = [d['label'] for d in frame_data]
        final_label = 1 if np.mean(frame_labels) > 0.4 else 0

        return coords, frame_data, {
            'avg_si': avg_si,
            'avg_left_knee': avg_left_knee,
            'avg_right_knee': avg_right_knee,
            'avg_knee': avg_knee,
            'video_name': os.path.basename(path),
            'final_label': final_label
        }

# =============================================================================
# LABELLING STRATEGY
# =============================================================================
def label_videos(all_video_metrics):
    video_labels = {}
    print(f"   Using fixed threshold: SI > {SI_THRESHOLD} = Bad")

    for m in all_video_metrics:
        video_labels[m['video_name']] = m['final_label']

    num_bad = sum(1 for v in video_labels.values() if v == 1)
    num_good = len(video_labels) - num_bad
    print(f"   Labels: {num_good} Good, {num_bad} Bad")

    return video_labels

# =============================================================================
# DATA EXTRACTION (with automatic cache invalidation on channel mismatch)
# =============================================================================
def extract_all_data():
    # Check if we can use the cache
    use_cache = not FORCE_REEXTRACT and os.path.exists(CACHE_FILE)
    cache_valid = False

    if use_cache:
        try:
            cache = torch.load(CACHE_FILE)
            # Check if cache has the required keys for split format
            if 'X_train' in cache:
                # New split format: check channel count from X_train
                if cache['X_train'].size(1) == 12:
                    cache_valid = True
                    print(f"\n✅ Loading cached features (split format) from {CACHE_FILE}")
                    X_train, y_train = cache['X_train'], cache['y_train']
                    X_val, y_val = cache['X_val'], cache['y_val']
                    X_test, y_test = cache['X_test'], cache['y_test']
                    num_classes = cache.get('num_classes', len(torch.unique(y_train)))
                    print(f"   Loaded {len(X_train)} train, {len(X_val)} val, {len(X_test)} test samples.")
                    return X_train, X_val, X_test, y_train, y_val, y_test, num_classes
                else:
                    print(f"   Cache has {cache['X_train'].size(1)} channels, expected 12. Re-extracting...")
            else:
                # Old format: has 'X' and 'y'
                if 'X' in cache and 'y' in cache:
                    if cache['X'].size(1) == 12:
                        print(f"\n✅ Loading cached features (old format) from {CACHE_FILE}")
                        X, y = cache['X'], cache['y']
                        num_classes = cache.get('num_classes', len(torch.unique(y)))
                        # We'll split it below
                        cache_valid = True
                        # We'll let the code proceed to split and save new cache
                    else:
                        print(f"   Cache has {cache['X'].size(1)} channels, expected 12. Re-extracting...")
                else:
                    print("   Cache missing required keys. Re-extracting...")
        except Exception as e:
            print(f"   Error loading cache: {e}. Re-extracting...")

    # If cache is not valid, perform full extraction
    if not cache_valid:
        print("\n🔄 Extracting features from videos (this may take a while)...")
        proc = SquatProcessor()
        X = []
        window_video_names = []
        all_video_metrics = []

        videos = glob.glob(os.path.join(VIDEO_DIR, "*.mp4")) + \
                 glob.glob(os.path.join(VIDEO_DIR, "*.avi")) + \
                 glob.glob(os.path.join(VIDEO_DIR, "*.mov"))

        if not videos:
            raise ValueError(f"No video files found in {VIDEO_DIR}")

        print(f"\nFound {len(videos)} videos. Processing...")

        for v in tqdm(videos):
            coords, frame_data, metrics = proc.process_video(v)
            if coords is not None:
                all_video_metrics.append(metrics)
                video_name = metrics['video_name']

                for i in range(0, len(coords) - SEQ_LEN, STRIDE):
                    window = coords[i:i+SEQ_LEN]
                    vel = np.diff(window, axis=0, append=window[-1:])
                    acc = np.diff(vel, axis=0, append=vel[-1:])
                    jerk = np.diff(acc, axis=0, append=acc[-1:])
                    combined = np.concatenate([window, vel, acc, jerk], axis=-1)
                    combined = combined.transpose(2, 0, 1)
                    X.append(combined)
                    window_video_names.append(video_name)

        if not X:
            raise ValueError("No valid data extracted. Check videos.")

        print(f"\n✅ Total windows extracted: {len(X)}")
        print(f"✅ Total videos processed: {len(all_video_metrics)}")

        video_labels = label_videos(all_video_metrics)
        y = [video_labels[name] for name in window_video_names]

        unique_labels = np.unique(y)
        print(f"\n📊 Label distribution: {dict(zip(*np.unique(y, return_counts=True)))}")

        X = torch.FloatTensor(np.array(X))
        y = torch.LongTensor(np.array(y))
        num_classes = len(unique_labels)

        print("\n" + "="*60)
        print("DATASET SUMMARY")
        print("="*60)
        print(f"Total windows: {len(X)}")
        print(f"Classes found: {num_classes}")
        for label in sorted(unique_labels):
            class_name = "Good" if label == 0 else "Bad"
            count = (y == label).sum().item()
            print(f"  {class_name}: {count} ({count/len(y)*100:.1f}%)")

        # Now split the data
        unique_labels_tensor = torch.unique(y)
        if len(unique_labels_tensor) < 2:
            print(f"\n⚠️  WARNING: Only {len(unique_labels_tensor)} class present!")
            print("   Using simple split (no stratification).")
            X_train, X_temp, y_train, y_temp = train_test_split(
                X, y, test_size=0.3, random_state=SEED
            )
            X_val, X_test, y_val, y_test = train_test_split(
                X_temp, y_temp, test_size=0.5, random_state=SEED
            )
        else:
            min_class_count = min([(y == label).sum().item() for label in unique_labels_tensor])
            if min_class_count < 2:
                print(f"\n⚠️  WARNING: One class has only {min_class_count} sample(s).")
                print("   Using simple split instead of stratified split.")
                X_train, X_temp, y_train, y_temp = train_test_split(
                    X, y, test_size=0.3, random_state=SEED
                )
                X_val, X_test, y_val, y_test = train_test_split(
                    X_temp, y_temp, test_size=0.5, random_state=SEED
                )
            else:
                X_train, X_temp, y_train, y_temp = train_test_split(
                    X, y, test_size=0.3, random_state=SEED, stratify=y
                )
                X_val, X_test, y_val, y_test = train_test_split(
                    X_temp, y_temp, test_size=0.5, random_state=SEED, stratify=y_temp
                )

        # Save the splits to cache
        os.makedirs(os.path.dirname(CACHE_FILE), exist_ok=True)
        torch.save({
            'X_train': X_train, 'y_train': y_train,
            'X_val': X_val, 'y_val': y_val,
            'X_test': X_test, 'y_test': y_test,
            'num_classes': num_classes
        }, CACHE_FILE)
        print(f"\n💾 Cached extracted features (with splits) to {CACHE_FILE}")

        return X_train, X_val, X_test, y_train, y_val, y_test, num_classes

    # If we reach here, cache was valid but old format (X,y) - we need to split
    # This part is only reached if cache_valid is True but we didn't return earlier (old format)
    # We'll load X,y from cache and split
    try:
        cache = torch.load(CACHE_FILE)
        X = cache['X']
        y = cache['y']
        num_classes = cache.get('num_classes', len(torch.unique(y)))
        # Proceed to split
        unique_labels_tensor = torch.unique(y)
        if len(unique_labels_tensor) < 2:
            print(f"\n⚠️  WARNING: Only {len(unique_labels_tensor)} class present!")
            print("   Using simple split (no stratification).")
            X_train, X_temp, y_train, y_temp = train_test_split(
                X, y, test_size=0.3, random_state=SEED
            )
            X_val, X_test, y_val, y_test = train_test_split(
                X_temp, y_temp, test_size=0.5, random_state=SEED
            )
        else:
            min_class_count = min([(y == label).sum().item() for label in unique_labels_tensor])
            if min_class_count < 2:
                print(f"\n⚠️  WARNING: One class has only {min_class_count} sample(s).")
                print("   Using simple split instead of stratified split.")
                X_train, X_temp, y_train, y_temp = train_test_split(
                    X, y, test_size=0.3, random_state=SEED
                )
                X_val, X_test, y_val, y_test = train_test_split(
                    X_temp, y_temp, test_size=0.5, random_state=SEED
                )
            else:
                X_train, X_temp, y_train, y_temp = train_test_split(
                    X, y, test_size=0.3, random_state=SEED, stratify=y
                )
                X_val, X_test, y_val, y_test = train_test_split(
                    X_temp, y_temp, test_size=0.5, random_state=SEED, stratify=y_temp
                )
        # Overwrite cache with split version
        torch.save({
            'X_train': X_train, 'y_train': y_train,
            'X_val': X_val, 'y_val': y_val,
            'X_test': X_test, 'y_test': y_test,
            'num_classes': num_classes
        }, CACHE_FILE)
        print(f"\n💾 Updated cache with splits to {CACHE_FILE}")
        return X_train, X_val, X_test, y_train, y_val, y_test, num_classes
    except Exception as e:
        print(f"Error during cache splitting: {e}. Re-extracting...")
        # Force re-extraction by recursion (but avoid infinite loop)
        return extract_all_data()  # This will re-extract because FORCE_REEXTRACT is False but cache will be invalid

# =============================================================================
# MODELS
# =============================================================================
class CTR_GC(nn.Module):
    def __init__(self, in_c, out_c, adj):
        super().__init__()
        self.adj = nn.Parameter(adj, requires_grad=False)
        self.refine = nn.Conv2d(in_c, out_c, 1)
        self.pa = nn.Parameter(torch.randn(out_c, 17, 17) * 0.01)
        self.alpha = nn.Parameter(torch.ones(1))

    def forward(self, x):
        x = self.refine(x)
        A = self.alpha * self.adj.unsqueeze(0) + self.pa
        x = torch.einsum('bctv,cvw->bctw', x, A)
        return x

class CTRGCN_Block(nn.Module):
    def __init__(self, in_c, out_c, adj, stride=1, dropout=0.6):
        super().__init__()
        self.gcn = CTR_GC(in_c, out_c, adj)
        self.tcn = nn.Sequential(
            nn.BatchNorm2d(out_c), nn.ReLU(inplace=True),
            nn.Conv2d(out_c, out_c, (9, 1), (stride, 1), (4, 0)),
            nn.BatchNorm2d(out_c), nn.Dropout(dropout)
        )
        self.res = nn.Conv2d(in_c, out_c, 1) if in_c != out_c else nn.Identity()
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        return self.relu(self.tcn(self.gcn(x)) + self.res(x))

class CTRGCN(nn.Module):
    def __init__(self, in_channels=12, num_classes=2):
        super().__init__()
        adj = get_adj_matrix()
        self.block1 = CTRGCN_Block(in_channels, 8, adj, dropout=0.6)
        self.block2 = CTRGCN_Block(8, 16, adj, dropout=0.6)
        self.block3 = CTRGCN_Block(16, 32, adj, dropout=0.7)
        self.block4 = CTRGCN_Block(32, 64, adj, dropout=0.7)
        self.block5 = CTRGCN_Block(64, 128, adj, dropout=0.8)
        self.block6 = CTRGCN_Block(128, 256, adj, dropout=0.8)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Dropout(0.8),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.block4(x)
        x = self.block5(x)
        x = self.block6(x)
        x = self.pool(x).view(x.size(0), -1)
        return self.fc(x)

class ST_GCN_Layer(nn.Module):
    def __init__(self, in_c, out_c, adj, stride=1, dropout=0.6):
        super().__init__()
        self.adj = nn.Parameter(adj, requires_grad=False)
        self.gcn = nn.Conv2d(in_c, out_c, kernel_size=1)
        self.tcn = nn.Sequential(
            nn.BatchNorm2d(out_c), nn.ReLU(inplace=True),
            nn.Conv2d(out_c, out_c, (9, 1), (stride, 1), (4, 0)),
            nn.BatchNorm2d(out_c), nn.Dropout(dropout)
        )
        self.residual = nn.Conv2d(in_c, out_c, 1) if in_c != out_c else nn.Identity()
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        res = self.residual(x)
        x = torch.einsum('nctv,vw->nctw', x, self.adj)
        x = self.gcn(x)
        x = self.tcn(x) + 0.6 * res
        return self.relu(x)

class STGCN(nn.Module):
    def __init__(self, in_channels=12, num_classes=2):
        super().__init__()
        adj = get_adj_matrix()
        self.layer1 = ST_GCN_Layer(in_channels, 8, adj, dropout=0.6)
        self.layer2 = ST_GCN_Layer(8, 16, adj, dropout=0.6)
        self.layer3 = ST_GCN_Layer(16, 32, adj, dropout=0.7)
        self.layer4 = ST_GCN_Layer(32, 64, adj, dropout=0.7)
        self.layer5 = ST_GCN_Layer(64, 128, adj, dropout=0.8)
        self.layer6 = ST_GCN_Layer(128, 256, adj, dropout=0.8)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Dropout(0.8),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.layer5(x)
        x = self.layer6(x)
        x = self.pool(x).view(x.size(0), -1)
        return self.fc(x)

# =============================================================================
# FOCAL LOSS
# =============================================================================
class FocalLoss(nn.Module):
    def __init__(self, alpha=0.25, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, reduction='none')
        pt = torch.exp(-ce_loss)
        alpha_t = self.alpha * targets + (1 - self.alpha) * (1 - targets)
        focal_loss = alpha_t * (1 - pt) ** self.gamma * ce_loss
        return focal_loss.mean()

# =============================================================================
# EVALUATION
# =============================================================================
def evaluate(model, loader, criterion=None):
    model.eval()
    all_preds, all_labels = [], []
    total_loss = 0
    with torch.no_grad():
        for data, target in loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            if criterion:
                loss = criterion(output, target)
                total_loss += loss.item()
            all_preds.extend(torch.argmax(output, 1).cpu().numpy())
            all_labels.extend(target.cpu().numpy())
    acc = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds, average='weighted')
    return acc, f1, total_loss/len(loader) if criterion else 0, all_labels, all_preds

# =============================================================================
# TRAINING FUNCTION
# =============================================================================
def train_model(model_name, train_loader, val_loader, test_loader, num_classes=2):
    if model_name == "CTR-GCN":
        model = CTRGCN(in_channels=12, num_classes=num_classes).to(device)
    else:
        model = STGCN(in_channels=12, num_classes=num_classes).to(device)
    print(f"\n{'='*50}\nTraining {model_name}...\n{'='*50}")

    criterion = FocalLoss(alpha=0.25, gamma=2.0)
    optimizer = Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)

    best_val_f1 = -1.0
    best_state = None
    patience_counter = 0
    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': [], 'val_f1': []}

    for epoch in range(EPOCHS):
        model.train()
        epoch_loss = 0
        correct_train = 0
        total_train = 0

        for data, target in train_loader:
            data = SkeletonUtils.augment_batch(data)
            data, target = data.to(device), target.to(device)

            optimizer.zero_grad()
            output = model(data)
            loss = criterion(output, target)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            epoch_loss += loss.item()
            _, pred = torch.max(output, 1)
            correct_train += (pred == target).sum().item()
            total_train += target.size(0)

        train_acc = correct_train / total_train
        val_acc, val_f1, val_loss, _, _ = evaluate(model, val_loader, criterion)

        history['train_loss'].append(epoch_loss/len(train_loader))
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)
        history['val_f1'].append(val_f1)

        scheduler.step()

        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1

        print(f"Epoch {epoch+1:03d} | Train Acc: {train_acc:.2%} | Val Acc: {val_acc:.2%} | Val F1: {val_f1:.4f}")

        if patience_counter >= PATIENCE and epoch > EARLY_STOP_MIN_EPOCH:
            print(f"Early stopping at epoch {epoch+1}")
            break

    if best_state is not None:
        model.load_state_dict(best_state)
    test_acc, test_f1, _, y_true, y_pred = evaluate(model, test_loader)
    print(f"\n{model_name} Test Accuracy: {test_acc:.4f} ({test_acc:.2%})")
    print(f"{model_name} Test Weighted F1: {test_f1:.4f}")
    print("\nClassification Report:")
    target_names = ['Good', 'Bad'][:num_classes]
    print(classification_report(y_true, y_pred, target_names=target_names))

    return test_acc, test_f1, history, y_true, y_pred

# =============================================================================
# CROSS-VALIDATION FUNCTION
# =============================================================================
def run_cross_validation(X_train, y_train, num_classes):
    """
    Run K-Fold cross-validation on the training data.
    Returns: average accuracy, average F1, and per-fold results
    """
    print("\n" + "="*70)
    print(f"CROSS-VALIDATION ({N_FOLDS}-FOLD)")
    print("="*70)
    
    # Combine train data into a single dataset
    train_data = torch.utils.data.TensorDataset(X_train, y_train)
    
    # K-Fold split
    kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
    
    cv_results = {}
    
    for model_name in MODELS_TO_TRAIN:
        print(f"\n\n{'#'*60}")
        print(f"# CROSS-VALIDATION FOR {model_name}")
        print(f"{'#'*60}")
        
        fold_accuracies = []
        fold_f1_scores = []
        fold_reports = []
        all_y_true = []
        all_y_pred = []
        
        for fold, (train_idx, val_idx) in enumerate(kf.split(train_data)):
            print(f"\n--- Fold {fold+1}/{N_FOLDS} ---")
            
            # Create subsets
            train_subset = Subset(train_data, train_idx)
            val_subset = Subset(train_data, val_idx)
            
            # Create data loaders
            train_loader = DataLoader(train_subset, BATCH_SIZE, shuffle=True)
            val_loader = DataLoader(val_subset, BATCH_SIZE, shuffle=False)
            
            # Create a dummy test loader (not used)
            test_loader = val_loader  # Use validation as test for this fold
            
            # Train model
            if model_name == "CTR-GCN":
                model = CTRGCN(in_channels=12, num_classes=num_classes).to(device)
            else:
                model = STGCN(in_channels=12, num_classes=num_classes).to(device)
            
            criterion = FocalLoss(alpha=0.25, gamma=2.0)
            optimizer = Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
            scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)
            
            best_val_f1 = -1.0
            best_state = None
            patience_counter = 0
            
            for epoch in range(EPOCHS):
                model.train()
                epoch_loss = 0
                correct_train = 0
                total_train = 0
                
                for data, target in train_loader:
                    data = SkeletonUtils.augment_batch(data)
                    data, target = data.to(device), target.to(device)
                    
                    optimizer.zero_grad()
                    output = model(data)
                    loss = criterion(output, target)
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                    optimizer.step()
                    
                    epoch_loss += loss.item()
                    _, pred = torch.max(output, 1)
                    correct_train += (pred == target).sum().item()
                    total_train += target.size(0)
                
                train_acc = correct_train / total_train
                val_acc, val_f1, val_loss, _, _ = evaluate(model, val_loader, criterion)
                scheduler.step()
                
                if val_f1 > best_val_f1:
                    best_val_f1 = val_f1
                    best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
                    patience_counter = 0
                else:
                    patience_counter += 1
                
                if patience_counter >= PATIENCE and epoch > EARLY_STOP_MIN_EPOCH:
                    break
            
            # Load best model and evaluate on validation set
            if best_state is not None:
                model.load_state_dict(best_state)
            
            val_acc, val_f1, _, y_true, y_pred = evaluate(model, val_loader)
            fold_accuracies.append(val_acc)
            fold_f1_scores.append(val_f1)
            
            print(f"Fold {fold+1} Accuracy: {val_acc:.4f} ({val_acc:.2%})")
            print(f"Fold {fold+1} F1: {val_f1:.4f}")
            
            all_y_true.extend(y_true)
            all_y_pred.extend(y_pred)
        
        # Calculate average metrics
        avg_acc = np.mean(fold_accuracies)
        avg_f1 = np.mean(fold_f1_scores)
        std_acc = np.std(fold_accuracies)
        std_f1 = np.std(fold_f1_scores)
        
        print(f"\n{'='*50}")
        print(f"{model_name} CROSS-VALIDATION RESULTS")
        print(f"{'='*50}")
        print(f"Fold Accuracies: {[f'{acc:.4f}' for acc in fold_accuracies]}")
        print(f"Average Accuracy: {avg_acc:.4f} (±{std_acc:.4f})")
        print(f"Average F1: {avg_f1:.4f} (±{std_f1:.4f})")
        print(f"\nOverall Classification Report:")
        target_names = ['Good', 'Bad'][:num_classes]
        print(classification_report(all_y_true, all_y_pred, target_names=target_names))
        
        cv_results[model_name] = {
            'fold_accuracies': fold_accuracies,
            'fold_f1': fold_f1_scores,
            'avg_accuracy': avg_acc,
            'avg_f1': avg_f1,
            'std_accuracy': std_acc,
            'std_f1': std_f1,
            'y_true': all_y_true,
            'y_pred': all_y_pred
        }
    
    return cv_results

# =============================================================================
# MAIN
# =============================================================================
def main():
    print("\n" + "="*70)
    print("SQUAT CLASSIFICATION - FIXED THRESHOLD (SI = 10)")
    print("="*70)
    print(f"Dataset path: {VIDEO_DIR}")
    print(f"SI Threshold: {SI_THRESHOLD} (Fixed)")
    print(f"Rule: SI > {SI_THRESHOLD} = Bad Squat")
    print(f"Input features: Position + Velocity + Acceleration + Jerk (12 channels)")
    print(f"Cross-validation: {'ON' if RUN_CV else 'OFF'}")

    # Load the data splits from cache
    X_train, X_val, X_test, y_train, y_val, y_test, num_classes = extract_all_data()

    print(f"\n📊 Data split:")
    print(f"  Train: {len(X_train)} samples")
    print(f"  Validation: {len(X_val)} samples")
    print(f"  Test: {len(X_test)} samples")

    if RUN_CV:
        # Run cross-validation on training data
        cv_results = run_cross_validation(X_train, y_train, num_classes)
        
        # Optionally, train final model on all training data and test on test set
        print("\n" + "="*70)
        print("TRAINING FINAL MODEL ON FULL TRAINING DATA")
        print("="*70)
        
        results = {}
        for model_name in MODELS_TO_TRAIN:
            # Combine train and validation for final training
            X_full_train = torch.cat([X_train, X_val], dim=0)
            y_full_train = torch.cat([y_train, y_val], dim=0)
            
            train_loader = DataLoader(
                list(zip(X_full_train, y_full_train)), BATCH_SIZE, shuffle=True
            )
            test_loader = DataLoader(
                list(zip(X_test, y_test)), BATCH_SIZE, shuffle=False
            )
            val_loader = DataLoader(
                list(zip(X_val, y_val)), BATCH_SIZE, shuffle=False
            )
            
            acc, f1, hist, y_true, y_pred = train_model(
                model_name, train_loader, val_loader, test_loader, num_classes
            )
            results[model_name] = {'accuracy': acc, 'f1': f1, 'history': hist, 'y_true': y_true, 'y_pred': y_pred}
        
        # Plot results
        fig, axes = plt.subplots(1, len(results), figsize=(7*len(results), 5), squeeze=False)
        for i, (name, res) in enumerate(results.items()):
            ax = axes[0][i]
            ax.plot(res['history']['train_acc'], label='Train Acc')
            ax.plot(res['history']['val_acc'], label='Val Acc')
            ax.set_title(f'{name} - Accuracy')
            ax.set_xlabel('Epoch')
            ax.set_ylabel('Accuracy')
            ax.legend()
            ax.grid(True)
        plt.tight_layout()
        plt.savefig(os.path.join(os.path.expanduser("~"), "Downloads", "Comparison_Accuracy.png"), dpi=300)
        plt.show()
        
        # Save CV results
        cv_summary = []
        for name, res in cv_results.items():
            cv_summary.append({
                'Model': name,
                'Avg Accuracy': res['avg_accuracy'],
                'Std Accuracy': res['std_accuracy'],
                'Avg F1': res['avg_f1'],
                'Std F1': res['std_f1'],
                'Fold Accuracies': str(res['fold_accuracies'])
            })
        cv_df = pd.DataFrame(cv_summary)
        cv_df.to_csv(os.path.join(os.path.expanduser("~"), "Downloads", "cv_results.csv"), index=False)
        print("\n📊 CV results saved to Downloads/cv_results.csv")
        
    else:
        # Single train-test split (original behavior)
        train_loader = DataLoader(
            list(zip(X_train, y_train)), BATCH_SIZE, shuffle=True
        )
        val_loader = DataLoader(
            list(zip(X_val, y_val)), BATCH_SIZE, shuffle=False
        )
        test_loader = DataLoader(
            list(zip(X_test, y_test)), BATCH_SIZE, shuffle=False
        )

        results = {}
        for model_name in MODELS_TO_TRAIN:
            acc, f1, hist, y_true, y_pred = train_model(
                model_name, train_loader, val_loader, test_loader, num_classes
            )
            results[model_name] = {'accuracy': acc, 'f1': f1, 'history': hist, 'y_true': y_true, 'y_pred': y_pred}

        print("\n" + "="*60)
        print("FINAL COMPARISON")
        print("="*60)
        for name, res in results.items():
            print(f"{name:10s} | Test Acc: {res['accuracy']:.4f} ({res['accuracy']:.2%}) | F1: {res['f1']:.4f}")

        fig, axes = plt.subplots(1, len(results), figsize=(7*len(results), 5), squeeze=False)
        for i, (name, res) in enumerate(results.items()):
            ax = axes[0][i]
            ax.plot(res['history']['train_acc'], label='Train Acc')
            ax.plot(res['history']['val_acc'], label='Val Acc')
            ax.set_title(f'{name} - Accuracy')
            ax.set_xlabel('Epoch')
            ax.set_ylabel('Accuracy')
            ax.legend()
            ax.grid(True)
        plt.tight_layout()
        plt.savefig(os.path.join(os.path.expanduser("~"), "Downloads", "Comparison_Accuracy.png"), dpi=300)
        plt.show()

        fig, axes = plt.subplots(1, len(results), figsize=(6*len(results), 5), squeeze=False)
        target_names = ['Good', 'Bad'][:num_classes]
        for i, (name, res) in enumerate(results.items()):
            cm = confusion_matrix(res['y_true'], res['y_pred'])
            sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                        xticklabels=target_names, yticklabels=target_names,
                        ax=axes[0][i])
            axes[0][i].set_title(f'{name} - Confusion Matrix')
            axes[0][i].set_ylabel('Actual')
            axes[0][i].set_xlabel('Predicted')
        plt.tight_layout()
        plt.savefig(os.path.join(os.path.expanduser("~"), "Downloads", "Comparison_Confusion.png"), dpi=300)
        plt.show()

    print("\n✅ All results saved to Downloads folder.")
    print("\n📁 Files saved:")
    print("   - Comparison_Accuracy.png")
    print("   - Comparison_Confusion.png")
    print("   - cv_results.csv (if cross-validation was run)")

if __name__ == "__main__":
    main()

Using device: cpu | torch threads set to 24

SQUAT CLASSIFICATION - FIXED THRESHOLD (SI = 10)
Dataset path: C:\Users\admin\Downloads\A Multi-View Raw Video Dataset of Seven Fitness Ex\A Multi-View Raw Video Dataset of Seven Fitness Ex\Dataset Exercise Quality-wise\squat\New folder
SI Threshold: 10.0 (Fixed)
Rule: SI > 10.0 = Bad Squat
Input features: Position + Velocity + Acceleration + Jerk (12 channels)
Cross-validation: ON

✅ Loading cached features (split format) from C:\Users\admin\Downloads\squat_features_cache_1 (1).pt
   Loaded 4244 train, 909 val, 910 test samples.

📊 Data split:
  Train: 4244 samples
  Validation: 909 samples
  Test: 910 samples

CROSS-VALIDATION (5-FOLD)


############################################################
# CROSS-VALIDATION FOR CTR-GCN
############################################################

--- Fold 1/5 ---


KeyboardInterrupt: 

In [1]:
import ssl
# --- Fix for Windows SSL/cert-store errors when MediaPipe tries to download a
# model file it doesn't have cached yet (SSLError: [ASN1: NOT_ENOUGH_DATA]).
# Tries certifi's certificate bundle first; falls back to an unverified
# context (only used for this local model download) if certifi is missing.
try:
    import certifi
    ssl._create_default_https_context = lambda: ssl.create_default_context(cafile=certifi.where())
except ImportError:
    ssl._create_default_https_context = ssl._create_unverified_context

import cv2
import mediapipe as mp
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from torch.optim import Adam
from torch.optim.lr_scheduler import CosineAnnealingLR
from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import confusion_matrix, accuracy_score, classification_report, f1_score
import os
import glob
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# =============================================================================
# CONFIGURATION
# =============================================================================
VIDEO_DIR = r"E:\25PHD1029\Peer\Videos\Kaggle"

# ===== FIXED LABELLING STRATEGY =====
LABEL_STRATEGY = "threshold"
SI_THRESHOLD = 10.0

# Other parameters
SEQ_LEN = 45
STRIDE = 10
BATCH_SIZE = 32
EPOCHS = 20
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 1e-3
SEED = 42
PATIENCE = 8
EARLY_STOP_MIN_EPOCH = 10

# Which model(s) to train
MODELS_TO_TRAIN = ["CTR-GCN", "ST-GCN"]

# Feature cache
CACHE_FILE = os.path.join(os.path.expanduser("~"), "Downloads", "squat_features_cache_1 (2).pt")
FORCE_REEXTRACT = False

# Cross-validation settings
RUN_CV = True          # Set to True to run cross-validation, False for single train
N_FOLDS = 5            # Number of folds for cross-validation

# Graph adjacency for 17 joints
ADJ = [
    (0,1),(0,2),(1,3),(2,4), (5,6),(5,7),(7,9),(6,8),(8,10),
    (11,12),(11,13),(13,15),(12,14),(14,16), (5,11),(6,12)
]

def set_seed(seed=SEED):
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

if device.type == 'cpu':
    n_threads = os.cpu_count() or 4
    torch.set_num_threads(n_threads)
    print(f"Using device: {device} | torch threads set to {n_threads}")
else:
    print(f"Using device: {device}")

def get_adj_matrix():
    A = np.eye(17)
    for i, j in ADJ:
        A[i, j] = A[j, i] = 1
    D = np.diag(np.sum(A, axis=1)**-0.5)
    A_norm = D @ A @ D
    return torch.FloatTensor(A_norm)

# =============================================================================
# SKELETON UTILS & PROCESSOR
# =============================================================================
class SkeletonUtils:
    @staticmethod
    def normalize(coords):
        mid_hip = (coords[:, 11, :] + coords[:, 12, :]) / 2
        coords = coords - mid_hip[:, np.newaxis, :]
        shoulder_mid = (coords[:, 0, :] + coords[:, 1, :]) / 2
        hip_mid = (coords[:, 11, :] + coords[:, 12, :]) / 2
        body_scale = np.linalg.norm(shoulder_mid - hip_mid, axis=-1).mean()
        if body_scale > 1e-6:
            coords = coords / body_scale
        mean = coords.mean(axis=0, keepdims=True)
        std = coords.std(axis=0, keepdims=True) + 1e-8
        coords = (coords - mean) / std
        return coords

    @staticmethod
    def calculate_angle(a, b, c):
        a, b, c = np.array(a), np.array(b), np.array(c)
        ba = a - b
        bc = c - b
        cos_angle = np.dot(ba, bc) / (np.linalg.norm(ba) * np.linalg.norm(bc) + 1e-8)
        angle = np.arccos(np.clip(cos_angle, -1.0, 1.0))
        return np.degrees(angle)

    @staticmethod
    def augment_batch(x):
        x = x.clone()
        B = x.size(0)
        pos = x[:, 0:3]
        vel = x[:, 3:6]
        acc = x[:, 6:9]
        jerk = x[:, 9:12]

        x += torch.randn_like(x) * 0.02

        do_rot = torch.rand(B) > 0.5
        if do_rot.any():
            angles = (torch.rand(B) - 0.5) * 0.3
            c, s = torch.cos(angles), torch.sin(angles)
            zeros, ones = torch.zeros(B), torch.ones(B)
            rot = torch.stack([
                torch.stack([c, zeros, s], dim=1),
                torch.stack([zeros, ones, zeros], dim=1),
                torch.stack([-s, zeros, c], dim=1),
            ], dim=1).to(x.device)

            idx = do_rot.nonzero(as_tuple=True)[0]
            pos_sub = pos[idx]
            vel_sub = vel[idx]
            acc_sub = acc[idx]
            jerk_sub = jerk[idx]
            b, _, T, V = pos_sub.shape
            pos_flat = pos_sub.view(b, 3, -1)
            vel_flat = vel_sub.view(b, 3, -1)
            acc_flat = acc_sub.view(b, 3, -1)
            jerk_flat = jerk_sub.view(b, 3, -1)

            rot_sub = rot[idx]
            pos_rot = torch.matmul(rot_sub, pos_flat)
            vel_rot = torch.matmul(rot_sub, vel_flat)
            acc_rot = torch.matmul(rot_sub, acc_flat)
            jerk_rot = torch.matmul(rot_sub, jerk_flat)

            pos[idx] = pos_rot.view(b, 3, T, V)
            vel[idx] = vel_rot.view(b, 3, T, V)
            acc[idx] = acc_rot.view(b, 3, T, V)
            jerk[idx] = jerk_rot.view(b, 3, T, V)

        x = torch.cat([pos, vel, acc, jerk], dim=1)
        scale = 1 + (torch.rand(B, 1, 1, 1) - 0.5) * 0.2
        x = x * scale
        return x

class SquatProcessor:
    def __init__(self):
        self.pose = mp.solutions.pose.Pose(
            static_image_mode=False,
            model_complexity=1,
            min_detection_confidence=0.5
        )

    def get_si(self, left_angle, right_angle):
        return ((left_angle - right_angle) / (0.5 * (left_angle + right_angle) + 1e-6)) * 100

    def process_video(self, path):
        cap = cv2.VideoCapture(path)
        coords = []
        frame_data = []

        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break
            img = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            res = self.pose.process(img)

            if res.pose_world_landmarks:
                wlm = res.pose_world_landmarks.landmark
                pts = [[lm.x, lm.y, lm.z] for lm in wlm[12:29]]
                coords.append(pts)

                left_knee = SkeletonUtils.calculate_angle(
                    [wlm[23].x, wlm[23].y, wlm[23].z],
                    [wlm[25].x, wlm[25].y, wlm[25].z],
                    [wlm[27].x, wlm[27].y, wlm[27].z]
                )
                right_knee = SkeletonUtils.calculate_angle(
                    [wlm[24].x, wlm[24].y, wlm[24].z],
                    [wlm[26].x, wlm[26].y, wlm[26].z],
                    [wlm[28].x, wlm[28].y, wlm[28].z]
                )
                si = self.get_si(left_knee, right_knee)

                frame_data.append({
                    'si': si,
                    'left_knee': left_knee,
                    'right_knee': right_knee,
                    'avg_knee': (left_knee + right_knee) / 2,
                    'label': 1 if si > SI_THRESHOLD else 0
                })

        cap.release()
        if len(coords) < SEQ_LEN:
            return None, None, None

        coords = SkeletonUtils.normalize(np.array(coords))

        avg_si = np.mean([d['si'] for d in frame_data])
        avg_left_knee = np.mean([d['left_knee'] for d in frame_data])
        avg_right_knee = np.mean([d['right_knee'] for d in frame_data])
        avg_knee = (avg_left_knee + avg_right_knee) / 2

        frame_labels = [d['label'] for d in frame_data]
        final_label = 1 if np.mean(frame_labels) > 0.4 else 0

        return coords, frame_data, {
            'avg_si': avg_si,
            'avg_left_knee': avg_left_knee,
            'avg_right_knee': avg_right_knee,
            'avg_knee': avg_knee,
            'video_name': os.path.basename(path),
            'final_label': final_label
        }

# =============================================================================
# LABELLING STRATEGY
# =============================================================================
def label_videos(all_video_metrics):
    video_labels = {}
    print(f"   Using fixed threshold: SI > {SI_THRESHOLD} = Bad")

    for m in all_video_metrics:
        video_labels[m['video_name']] = m['final_label']

    num_bad = sum(1 for v in video_labels.values() if v == 1)
    num_good = len(video_labels) - num_bad
    print(f"   Labels: {num_good} Good, {num_bad} Bad")

    return video_labels

# =============================================================================
# DATA EXTRACTION (with automatic cache invalidation on channel mismatch)
# =============================================================================
def extract_all_data():
    # Check if we can use the cache
    use_cache = not FORCE_REEXTRACT and os.path.exists(CACHE_FILE)
    cache_valid = False

    if use_cache:
        try:
            cache = torch.load(CACHE_FILE)
            # Check if cache has the required keys for split format
            if 'X_train' in cache:
                # New split format: check channel count from X_train
                if cache['X_train'].size(1) == 12:
                    cache_valid = True
                    print(f"\n✅ Loading cached features (split format) from {CACHE_FILE}")
                    X_train, y_train = cache['X_train'], cache['y_train']
                    X_val, y_val = cache['X_val'], cache['y_val']
                    X_test, y_test = cache['X_test'], cache['y_test']
                    num_classes = cache.get('num_classes', len(torch.unique(y_train)))
                    print(f"   Loaded {len(X_train)} train, {len(X_val)} val, {len(X_test)} test samples.")
                    return X_train, X_val, X_test, y_train, y_val, y_test, num_classes
                else:
                    print(f"   Cache has {cache['X_train'].size(1)} channels, expected 12. Re-extracting...")
            else:
                # Old format: has 'X' and 'y'
                if 'X' in cache and 'y' in cache:
                    if cache['X'].size(1) == 12:
                        print(f"\n✅ Loading cached features (old format) from {CACHE_FILE}")
                        X, y = cache['X'], cache['y']
                        num_classes = cache.get('num_classes', len(torch.unique(y)))
                        # We'll split it below
                        cache_valid = True
                        # We'll let the code proceed to split and save new cache
                    else:
                        print(f"   Cache has {cache['X'].size(1)} channels, expected 12. Re-extracting...")
                else:
                    print("   Cache missing required keys. Re-extracting...")
        except Exception as e:
            print(f"   Error loading cache: {e}. Re-extracting...")

    # If cache is not valid, perform full extraction
    if not cache_valid:
        print("\n🔄 Extracting features from videos (this may take a while)...")
        proc = SquatProcessor()
        X = []
        window_video_names = []
        all_video_metrics = []

        videos = glob.glob(os.path.join(VIDEO_DIR, "*.mp4")) + \
                 glob.glob(os.path.join(VIDEO_DIR, "*.avi")) + \
                 glob.glob(os.path.join(VIDEO_DIR, "*.mov"))

        if not videos:
            raise ValueError(f"No video files found in {VIDEO_DIR}")

        print(f"\nFound {len(videos)} videos. Processing...")

        for v in tqdm(videos):
            coords, frame_data, metrics = proc.process_video(v)
            if coords is not None:
                all_video_metrics.append(metrics)
                video_name = metrics['video_name']

                for i in range(0, len(coords) - SEQ_LEN, STRIDE):
                    window = coords[i:i+SEQ_LEN]
                    vel = np.diff(window, axis=0, append=window[-1:])
                    acc = np.diff(vel, axis=0, append=vel[-1:])
                    jerk = np.diff(acc, axis=0, append=acc[-1:])
                    combined = np.concatenate([window, vel, acc, jerk], axis=-1)
                    combined = combined.transpose(2, 0, 1)
                    X.append(combined)
                    window_video_names.append(video_name)

        if not X:
            raise ValueError("No valid data extracted. Check videos.")

        print(f"\n✅ Total windows extracted: {len(X)}")
        print(f"✅ Total videos processed: {len(all_video_metrics)}")

        video_labels = label_videos(all_video_metrics)
        y = [video_labels[name] for name in window_video_names]

        unique_labels = np.unique(y)
        print(f"\n📊 Label distribution: {dict(zip(*np.unique(y, return_counts=True)))}")

        X = torch.FloatTensor(np.array(X))
        y = torch.LongTensor(np.array(y))
        num_classes = len(unique_labels)

        print("\n" + "="*60)
        print("DATASET SUMMARY")
        print("="*60)
        print(f"Total windows: {len(X)}")
        print(f"Classes found: {num_classes}")
        for label in sorted(unique_labels):
            class_name = "Good" if label == 0 else "Bad"
            count = (y == label).sum().item()
            print(f"  {class_name}: {count} ({count/len(y)*100:.1f}%)")

        # Now split the data
        unique_labels_tensor = torch.unique(y)
        if len(unique_labels_tensor) < 2:
            print(f"\n⚠️  WARNING: Only {len(unique_labels_tensor)} class present!")
            print("   Using simple split (no stratification).")
            X_train, X_temp, y_train, y_temp = train_test_split(
                X, y, test_size=0.3, random_state=SEED
            )
            X_val, X_test, y_val, y_test = train_test_split(
                X_temp, y_temp, test_size=0.5, random_state=SEED
            )
        else:
            min_class_count = min([(y == label).sum().item() for label in unique_labels_tensor])
            if min_class_count < 2:
                print(f"\n⚠️  WARNING: One class has only {min_class_count} sample(s).")
                print("   Using simple split instead of stratified split.")
                X_train, X_temp, y_train, y_temp = train_test_split(
                    X, y, test_size=0.3, random_state=SEED
                )
                X_val, X_test, y_val, y_test = train_test_split(
                    X_temp, y_temp, test_size=0.5, random_state=SEED
                )
            else:
                X_train, X_temp, y_train, y_temp = train_test_split(
                    X, y, test_size=0.3, random_state=SEED, stratify=y
                )
                X_val, X_test, y_val, y_test = train_test_split(
                    X_temp, y_temp, test_size=0.5, random_state=SEED, stratify=y_temp
                )

        # Save the splits to cache
        os.makedirs(os.path.dirname(CACHE_FILE), exist_ok=True)
        torch.save({
            'X_train': X_train, 'y_train': y_train,
            'X_val': X_val, 'y_val': y_val,
            'X_test': X_test, 'y_test': y_test,
            'num_classes': num_classes
        }, CACHE_FILE)
        print(f"\n💾 Cached extracted features (with splits) to {CACHE_FILE}")

        return X_train, X_val, X_test, y_train, y_val, y_test, num_classes

    # If we reach here, cache was valid but old format (X,y) - we need to split
    # This part is only reached if cache_valid is True but we didn't return earlier (old format)
    # We'll load X,y from cache and split
    try:
        cache = torch.load(CACHE_FILE)
        X = cache['X']
        y = cache['y']
        num_classes = cache.get('num_classes', len(torch.unique(y)))
        # Proceed to split
        unique_labels_tensor = torch.unique(y)
        if len(unique_labels_tensor) < 2:
            print(f"\n⚠️  WARNING: Only {len(unique_labels_tensor)} class present!")
            print("   Using simple split (no stratification).")
            X_train, X_temp, y_train, y_temp = train_test_split(
                X, y, test_size=0.3, random_state=SEED
            )
            X_val, X_test, y_val, y_test = train_test_split(
                X_temp, y_temp, test_size=0.5, random_state=SEED
            )
        else:
            min_class_count = min([(y == label).sum().item() for label in unique_labels_tensor])
            if min_class_count < 2:
                print(f"\n⚠️  WARNING: One class has only {min_class_count} sample(s).")
                print("   Using simple split instead of stratified split.")
                X_train, X_temp, y_train, y_temp = train_test_split(
                    X, y, test_size=0.3, random_state=SEED
                )
                X_val, X_test, y_val, y_test = train_test_split(
                    X_temp, y_temp, test_size=0.5, random_state=SEED
                )
            else:
                X_train, X_temp, y_train, y_temp = train_test_split(
                    X, y, test_size=0.3, random_state=SEED, stratify=y
                )
                X_val, X_test, y_val, y_test = train_test_split(
                    X_temp, y_temp, test_size=0.5, random_state=SEED, stratify=y_temp
                )
        # Overwrite cache with split version
        torch.save({
            'X_train': X_train, 'y_train': y_train,
            'X_val': X_val, 'y_val': y_val,
            'X_test': X_test, 'y_test': y_test,
            'num_classes': num_classes
        }, CACHE_FILE)
        print(f"\n💾 Updated cache with splits to {CACHE_FILE}")
        return X_train, X_val, X_test, y_train, y_val, y_test, num_classes
    except Exception as e:
        print(f"Error during cache splitting: {e}. Re-extracting...")
        # Force re-extraction by recursion (but avoid infinite loop)
        return extract_all_data()  # This will re-extract because FORCE_REEXTRACT is False but cache will be invalid

# =============================================================================
# MODELS
# =============================================================================
class CTR_GC(nn.Module):
    def __init__(self, in_c, out_c, adj):
        super().__init__()
        self.adj = nn.Parameter(adj, requires_grad=False)
        self.refine = nn.Conv2d(in_c, out_c, 1)
        self.pa = nn.Parameter(torch.randn(out_c, 17, 17) * 0.01)
        self.alpha = nn.Parameter(torch.ones(1))

    def forward(self, x):
        x = self.refine(x)
        A = self.alpha * self.adj.unsqueeze(0) + self.pa
        x = torch.einsum('bctv,cvw->bctw', x, A)
        return x

class CTRGCN_Block(nn.Module):
    def __init__(self, in_c, out_c, adj, stride=1, dropout=0.6):
        super().__init__()
        self.gcn = CTR_GC(in_c, out_c, adj)
        self.tcn = nn.Sequential(
            nn.BatchNorm2d(out_c), nn.ReLU(inplace=True),
            nn.Conv2d(out_c, out_c, (9, 1), (stride, 1), (4, 0)),
            nn.BatchNorm2d(out_c), nn.Dropout(dropout)
        )
        self.res = nn.Conv2d(in_c, out_c, 1) if in_c != out_c else nn.Identity()
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        return self.relu(self.tcn(self.gcn(x)) + self.res(x))

class CTRGCN(nn.Module):
    def __init__(self, in_channels=12, num_classes=2):
        super().__init__()
        adj = get_adj_matrix()
        self.block1 = CTRGCN_Block(in_channels, 8, adj, dropout=0.6)
        self.block2 = CTRGCN_Block(8, 16, adj, dropout=0.6)
        self.block3 = CTRGCN_Block(16, 32, adj, dropout=0.7)
        self.block4 = CTRGCN_Block(32, 64, adj, dropout=0.7)
        self.block5 = CTRGCN_Block(64, 128, adj, dropout=0.8)
        self.block6 = CTRGCN_Block(128, 256, adj, dropout=0.8)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Dropout(0.8),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.block4(x)
        x = self.block5(x)
        x = self.block6(x)
        x = self.pool(x).view(x.size(0), -1)
        return self.fc(x)

class ST_GCN_Layer(nn.Module):
    def __init__(self, in_c, out_c, adj, stride=1, dropout=0.6):
        super().__init__()
        self.adj = nn.Parameter(adj, requires_grad=False)
        self.gcn = nn.Conv2d(in_c, out_c, kernel_size=1)
        self.tcn = nn.Sequential(
            nn.BatchNorm2d(out_c), nn.ReLU(inplace=True),
            nn.Conv2d(out_c, out_c, (9, 1), (stride, 1), (4, 0)),
            nn.BatchNorm2d(out_c), nn.Dropout(dropout)
        )
        self.residual = nn.Conv2d(in_c, out_c, 1) if in_c != out_c else nn.Identity()
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        res = self.residual(x)
        x = torch.einsum('nctv,vw->nctw', x, self.adj)
        x = self.gcn(x)
        x = self.tcn(x) + 0.6 * res
        return self.relu(x)

class STGCN(nn.Module):
    def __init__(self, in_channels=12, num_classes=2):
        super().__init__()
        adj = get_adj_matrix()
        self.layer1 = ST_GCN_Layer(in_channels, 8, adj, dropout=0.6)
        self.layer2 = ST_GCN_Layer(8, 16, adj, dropout=0.6)
        self.layer3 = ST_GCN_Layer(16, 32, adj, dropout=0.7)
        self.layer4 = ST_GCN_Layer(32, 64, adj, dropout=0.7)
        self.layer5 = ST_GCN_Layer(64, 128, adj, dropout=0.8)
        self.layer6 = ST_GCN_Layer(128, 256, adj, dropout=0.8)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Dropout(0.8),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.layer5(x)
        x = self.layer6(x)
        x = self.pool(x).view(x.size(0), -1)
        return self.fc(x)

# =============================================================================
# FOCAL LOSS
# =============================================================================
class FocalLoss(nn.Module):
    def __init__(self, alpha=0.25, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, reduction='none')
        pt = torch.exp(-ce_loss)
        alpha_t = self.alpha * targets + (1 - self.alpha) * (1 - targets)
        focal_loss = alpha_t * (1 - pt) ** self.gamma * ce_loss
        return focal_loss.mean()

# =============================================================================
# EVALUATION
# =============================================================================
def evaluate(model, loader, criterion=None):
    model.eval()
    all_preds, all_labels = [], []
    total_loss = 0
    with torch.no_grad():
        for data, target in loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            if criterion:
                loss = criterion(output, target)
                total_loss += loss.item()
            all_preds.extend(torch.argmax(output, 1).cpu().numpy())
            all_labels.extend(target.cpu().numpy())
    acc = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds, average='weighted')
    return acc, f1, total_loss/len(loader) if criterion else 0, all_labels, all_preds

# =============================================================================
# TRAINING FUNCTION (used only in single‑split mode now)
# =============================================================================
def train_model(model_name, train_loader, val_loader, test_loader, num_classes=2):
    if model_name == "CTR-GCN":
        model = CTRGCN(in_channels=12, num_classes=num_classes).to(device)
    else:
        model = STGCN(in_channels=12, num_classes=num_classes).to(device)
    print(f"\n{'='*50}\nTraining {model_name}...\n{'='*50}")

    criterion = FocalLoss(alpha=0.25, gamma=2.0)
    optimizer = Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)

    best_val_f1 = -1.0
    best_state = None
    patience_counter = 0
    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': [], 'val_f1': []}

    for epoch in range(EPOCHS):
        model.train()
        epoch_loss = 0
        correct_train = 0
        total_train = 0

        for data, target in train_loader:
            data = SkeletonUtils.augment_batch(data)
            data, target = data.to(device), target.to(device)

            optimizer.zero_grad()
            output = model(data)
            loss = criterion(output, target)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            epoch_loss += loss.item()
            _, pred = torch.max(output, 1)
            correct_train += (pred == target).sum().item()
            total_train += target.size(0)

        train_acc = correct_train / total_train
        val_acc, val_f1, val_loss, _, _ = evaluate(model, val_loader, criterion)

        history['train_loss'].append(epoch_loss/len(train_loader))
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)
        history['val_f1'].append(val_f1)

        scheduler.step()

        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1

        print(f"Epoch {epoch+1:03d} | Train Acc: {train_acc:.2%} | Val Acc: {val_acc:.2%} | Val F1: {val_f1:.4f}")

        if patience_counter >= PATIENCE and epoch > EARLY_STOP_MIN_EPOCH:
            print(f"Early stopping at epoch {epoch+1}")
            break

    if best_state is not None:
        model.load_state_dict(best_state)
    test_acc, test_f1, _, y_true, y_pred = evaluate(model, test_loader)
    print(f"\n{model_name} Test Accuracy: {test_acc:.4f} ({test_acc:.2%})")
    print(f"{model_name} Test Weighted F1: {test_f1:.4f}")
    print("\nClassification Report:")
    target_names = ['Good', 'Bad'][:num_classes]
    print(classification_report(y_true, y_pred, target_names=target_names))

    return test_acc, test_f1, history, y_true, y_pred

# =============================================================================
# CROSS-VALIDATION FUNCTION
# =============================================================================
def run_cross_validation(X_train, y_train, num_classes):
    """
    Run K-Fold cross-validation on the training data.
    Returns: average accuracy, average F1, and per-fold results
    """
    print("\n" + "="*70)
    print(f"CROSS-VALIDATION ({N_FOLDS}-FOLD)")
    print("="*70)
    
    # Combine train data into a single dataset
    train_data = torch.utils.data.TensorDataset(X_train, y_train)
    
    # K-Fold split
    kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
    
    cv_results = {}
    
    for model_name in MODELS_TO_TRAIN:
        print(f"\n\n{'#'*60}")
        print(f"# CROSS-VALIDATION FOR {model_name}")
        print(f"{'#'*60}")
        
        fold_accuracies = []
        fold_f1_scores = []
        fold_reports = []
        all_y_true = []
        all_y_pred = []
        
        for fold, (train_idx, val_idx) in enumerate(kf.split(train_data)):
            print(f"\n--- Fold {fold+1}/{N_FOLDS} ---")
            
            # Create subsets
            train_subset = Subset(train_data, train_idx)
            val_subset = Subset(train_data, val_idx)
            
            # Create data loaders
            train_loader = DataLoader(train_subset, BATCH_SIZE, shuffle=True)
            val_loader = DataLoader(val_subset, BATCH_SIZE, shuffle=False)
            
            # Create a dummy test loader (not used)
            test_loader = val_loader  # Use validation as test for this fold
            
            # Train model
            if model_name == "CTR-GCN":
                model = CTRGCN(in_channels=12, num_classes=num_classes).to(device)
            else:
                model = STGCN(in_channels=12, num_classes=num_classes).to(device)
            
            criterion = FocalLoss(alpha=0.25, gamma=2.0)
            optimizer = Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
            scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)
            
            best_val_f1 = -1.0
            best_state = None
            patience_counter = 0
            
            for epoch in range(EPOCHS):
                model.train()
                epoch_loss = 0
                correct_train = 0
                total_train = 0
                
                for data, target in train_loader:
                    data = SkeletonUtils.augment_batch(data)
                    data, target = data.to(device), target.to(device)
                    
                    optimizer.zero_grad()
                    output = model(data)
                    loss = criterion(output, target)
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                    optimizer.step()
                    
                    epoch_loss += loss.item()
                    _, pred = torch.max(output, 1)
                    correct_train += (pred == target).sum().item()
                    total_train += target.size(0)
                
                train_acc = correct_train / total_train
                val_acc, val_f1, val_loss, _, _ = evaluate(model, val_loader, criterion)
                scheduler.step()
                
                if val_f1 > best_val_f1:
                    best_val_f1 = val_f1
                    best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
                    patience_counter = 0
                else:
                    patience_counter += 1
                
                if patience_counter >= PATIENCE and epoch > EARLY_STOP_MIN_EPOCH:
                    break
            
            # Load best model and evaluate on validation set
            if best_state is not None:
                model.load_state_dict(best_state)
            
            val_acc, val_f1, _, y_true, y_pred = evaluate(model, val_loader)
            fold_accuracies.append(val_acc)
            fold_f1_scores.append(val_f1)
            
            print(f"Fold {fold+1} Accuracy: {val_acc:.4f} ({val_acc:.2%})")
            print(f"Fold {fold+1} F1: {val_f1:.4f}")
            
            all_y_true.extend(y_true)
            all_y_pred.extend(y_pred)
        
        # Calculate average metrics
        avg_acc = np.mean(fold_accuracies)
        avg_f1 = np.mean(fold_f1_scores)
        std_acc = np.std(fold_accuracies)
        std_f1 = np.std(fold_f1_scores)
        
        print(f"\n{'='*50}")
        print(f"{model_name} CROSS-VALIDATION RESULTS")
        print(f"{'='*50}")
        print(f"Fold Accuracies: {[f'{acc:.4f}' for acc in fold_accuracies]}")
        print(f"Average Accuracy: {avg_acc:.4f} (±{std_acc:.4f})")
        print(f"Average F1: {avg_f1:.4f} (±{std_f1:.4f})")
        print(f"\nOverall Classification Report:")
        target_names = ['Good', 'Bad'][:num_classes]
        print(classification_report(all_y_true, all_y_pred, target_names=target_names))
        
        cv_results[model_name] = {
            'fold_accuracies': fold_accuracies,
            'fold_f1': fold_f1_scores,
            'avg_accuracy': avg_acc,
            'avg_f1': avg_f1,
            'std_accuracy': std_acc,
            'std_f1': std_f1,
            'y_true': all_y_true,
            'y_pred': all_y_pred
        }
    
    return cv_results

# =============================================================================
# MAIN
# =============================================================================
def main():
    print("\n" + "="*70)
    print("SQUAT CLASSIFICATION - FIXED THRESHOLD (SI = 10)")
    print("="*70)
    print(f"Dataset path: {VIDEO_DIR}")
    print(f"SI Threshold: {SI_THRESHOLD} (Fixed)")
    print(f"Rule: SI > {SI_THRESHOLD} = Bad Squat")
    print(f"Input features: Position + Velocity + Acceleration + Jerk (12 channels)")
    print(f"Cross-validation: {'ON' if RUN_CV else 'OFF'}")

    # Load the data splits from cache
    X_train, X_val, X_test, y_train, y_val, y_test, num_classes = extract_all_data()

    print(f"\n📊 Data split:")
    print(f"  Train: {len(X_train)} samples")
    print(f"  Validation: {len(X_val)} samples")
    print(f"  Test: {len(X_test)} samples")

    if RUN_CV:
        # Run cross-validation on training data
        cv_results = run_cross_validation(X_train, y_train, num_classes)
        
        # Save CV results summary
        cv_summary = []
        for name, res in cv_results.items():
            cv_summary.append({
                'Model': name,
                'Avg Accuracy': res['avg_accuracy'],
                'Std Accuracy': res['std_accuracy'],
                'Avg F1': res['avg_f1'],
                'Std F1': res['std_f1'],
                'Fold Accuracies': str(res['fold_accuracies'])
            })
        cv_df = pd.DataFrame(cv_summary)
        cv_df.to_csv(os.path.join(os.path.expanduser("~"), "Downloads", "cv_results.csv"), index=False)
        print("\n📊 CV results saved to Downloads/cv_results.csv")
        print("\n✅ Cross-validation complete. No final model training or plots were generated.")
        # Exit after CV
        return

    else:
        # Single train-test split (original behavior) – kept for completeness
        train_loader = DataLoader(
            list(zip(X_train, y_train)), BATCH_SIZE, shuffle=True
        )
        val_loader = DataLoader(
            list(zip(X_val, y_val)), BATCH_SIZE, shuffle=False
        )
        test_loader = DataLoader(
            list(zip(X_test, y_test)), BATCH_SIZE, shuffle=False
        )

        results = {}
        for model_name in MODELS_TO_TRAIN:
            acc, f1, hist, y_true, y_pred = train_model(
                model_name, train_loader, val_loader, test_loader, num_classes
            )
            results[model_name] = {'accuracy': acc, 'f1': f1, 'history': hist, 'y_true': y_true, 'y_pred': y_pred}

        print("\n" + "="*60)
        print("FINAL COMPARISON")
        print("="*60)
        for name, res in results.items():
            print(f"{name:10s} | Test Acc: {res['accuracy']:.4f} ({res['accuracy']:.2%}) | F1: {res['f1']:.4f}")

        fig, axes = plt.subplots(1, len(results), figsize=(7*len(results), 5), squeeze=False)
        for i, (name, res) in enumerate(results.items()):
            ax = axes[0][i]
            ax.plot(res['history']['train_acc'], label='Train Acc')
            ax.plot(res['history']['val_acc'], label='Val Acc')
            ax.set_title(f'{name} - Accuracy')
            ax.set_xlabel('Epoch')
            ax.set_ylabel('Accuracy')
            ax.legend()
            ax.grid(True)
        plt.tight_layout()
        plt.savefig(os.path.join(os.path.expanduser("~"), "Downloads", "Comparison_Accuracy.png"), dpi=300)
        plt.show()

        fig, axes = plt.subplots(1, len(results), figsize=(6*len(results), 5), squeeze=False)
        target_names = ['Good', 'Bad'][:num_classes]
        for i, (name, res) in enumerate(results.items()):
            cm = confusion_matrix(res['y_true'], res['y_pred'])
            sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                        xticklabels=target_names, yticklabels=target_names,
                        ax=axes[0][i])
            axes[0][i].set_title(f'{name} - Confusion Matrix')
            axes[0][i].set_ylabel('Actual')
            axes[0][i].set_xlabel('Predicted')
        plt.tight_layout()
        plt.savefig(os.path.join(os.path.expanduser("~"), "Downloads", "Comparison_Confusion.png"), dpi=300)
        plt.show()

        print("\n✅ All results saved to Downloads folder.")
        print("\n📁 Files saved:")
        print("   - Comparison_Accuracy.png")
        print("   - Comparison_Confusion.png")

if __name__ == "__main__":
    main()

Using device: cpu | torch threads set to 24

SQUAT CLASSIFICATION - FIXED THRESHOLD (SI = 10)
Dataset path: E:\25PHD1029\Peer\Videos\Kaggle
SI Threshold: 10.0 (Fixed)
Rule: SI > 10.0 = Bad Squat
Input features: Position + Velocity + Acceleration + Jerk (12 channels)
Cross-validation: ON

✅ Loading cached features (old format) from C:\Users\admin\Downloads\squat_features_cache_1 (2).pt

💾 Updated cache with splits to C:\Users\admin\Downloads\squat_features_cache_1 (2).pt

📊 Data split:
  Train: 92 samples
  Validation: 20 samples
  Test: 20 samples

CROSS-VALIDATION (5-FOLD)


############################################################
# CROSS-VALIDATION FOR CTR-GCN
############################################################

--- Fold 1/5 ---
Fold 1 Accuracy: 0.5789 (57.89%)
Fold 1 F1: 0.4246

--- Fold 2/5 ---
Fold 2 Accuracy: 0.7368 (73.68%)
Fold 2 F1: 0.6252

--- Fold 3/5 ---
Fold 3 Accuracy: 0.7222 (72.22%)
Fold 3 F1: 0.6057

--- Fold 4/5 ---
Fold 4 Accuracy: 0.9444 (94.44%)
Fold 4